In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 200)

In [ ]:
INPUT_BASE = Path('/content/drive/MyDrive/Research')
BASE = Path('/content/drive/MyDrive/version_2')
DRIVE_ROOT = Path('/content/drive/MyDrive')


def resolve_file(*patterns):
    search_roots = [INPUT_BASE, BASE, DRIVE_ROOT]
    seen = set()
    for root in search_roots:
        if root in seen or not root.exists():
            continue
        seen.add(root)
        for pattern in patterns:
            matches = sorted(root.rglob(pattern))
            if matches:
                print(f'Resolved {pattern}: {matches[0]}')
                return matches[0]
    raise FileNotFoundError(patterns)


def stage_local_file(src_path, cache_dir=Path('/content/ds340w_cache')):
    cache_dir.mkdir(parents=True, exist_ok=True)
    dst_path = cache_dir / Path(src_path).name
    try:
        same_size = dst_path.exists() and dst_path.stat().st_size == Path(src_path).stat().st_size
    except OSError:
        same_size = False
    if same_size:
        print(f'Using cached local file: {dst_path}')
        return dst_path
    print(f'Copying to local runtime storage: {src_path} -> {dst_path}')
    shutil.copy2(src_path, dst_path)
    print(f'Local copy ready: {dst_path}')
    return dst_path


DATA_PATH = resolve_file('Updated_Crime_Enriched_PointLevel_v16c.csv', '*v16c*.csv')
LOCAL_DATA_PATH = stage_local_file(DATA_PATH)
OUT_DIR = BASE / 'methodology_data_summary_v16c'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_PATH =', DATA_PATH)
print('LOCAL_DATA_PATH =', LOCAL_DATA_PATH)
print('OUT_DIR =', OUT_DIR)

In [ ]:
df = pd.read_csv(LOCAL_DATA_PATH, low_memory=False)
print('Loaded shape:', df.shape)

In [ ]:
for col in ['crime_date', 'crime_time', 'occurred_dt', 'cmplnt_fr_dt', 'cmplnt_fr_tm', 'cmplnt_fr_tm_raw']:
    if col in df.columns:
        try:
            df[col] = pd.to_datetime(df[col], errors='coerce')
        except Exception:
            pass

numeric_preview_cols = [c for c in ['row_id', 'cmplnt_num', 'latitude', 'longitude', 'event_match_count', 'road_closure_count'] if c in df.columns]
for col in numeric_preview_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
overview_rows = []
overview_rows.append({'metric': 'total_rows', 'value': int(df.shape[0])})
overview_rows.append({'metric': 'total_columns', 'value': int(df.shape[1])})

if 'row_id' in df.columns:
    overview_rows.append({'metric': 'placeholder_row_count_row_id_0', 'value': int((df['row_id'] == 0).fillna(False).sum())})

if 'crime_date' in df.columns:
    overview_rows.append({'metric': 'crime_date_min', 'value': str(df['crime_date'].min())})
    overview_rows.append({'metric': 'crime_date_max', 'value': str(df['crime_date'].max())})

if 'occurred_dt' in df.columns:
    overview_rows.append({'metric': 'occurred_dt_non_missing', 'value': int(df['occurred_dt'].notna().sum())})

if 'crime_time' in df.columns:
    overview_rows.append({'metric': 'crime_time_non_missing', 'value': int(df['crime_time'].notna().sum())})

if 'cmplnt_fr_tm_raw' in df.columns:
    overview_rows.append({'metric': 'cmplnt_fr_tm_raw_non_missing', 'value': int(df['cmplnt_fr_tm_raw'].notna().sum())})

if 'cmplnt_fr_tm' in df.columns:
    overview_rows.append({'metric': 'cmplnt_fr_tm_non_missing', 'value': int(df['cmplnt_fr_tm'].notna().sum())})

if 'latitude' in df.columns:
    overview_rows.append({'metric': 'latitude_non_missing', 'value': int(df['latitude'].notna().sum())})

if 'longitude' in df.columns:
    overview_rows.append({'metric': 'longitude_non_missing', 'value': int(df['longitude'].notna().sum())})

if 'crime_time_source' in df.columns:
    source_counts = df['crime_time_source'].fillna('missing').value_counts(dropna=False)
    for key, value in source_counts.items():
        overview_rows.append({'metric': f'crime_time_source__{key}', 'value': int(value)})

overview_df = pd.DataFrame(overview_rows)
overview_df

In [ ]:
column_summary = pd.DataFrame({
    'variable': df.columns,
    'dtype': [str(df[c].dtype) for c in df.columns],
    'non_missing_count': [int(df[c].notna().sum()) for c in df.columns],
    'missing_count': [int(df[c].isna().sum()) for c in df.columns],
    'missing_pct': [round(float(df[c].isna().mean() * 100), 4) for c in df.columns],
    'unique_count': [int(df[c].nunique(dropna=True)) for c in df.columns],
    'example_1': [str(df[c].dropna().iloc[0]) if df[c].notna().any() else '' for c in df.columns],
    'example_2': [str(df[c].dropna().iloc[min(1, df[c].dropna().shape[0] - 1)]) if df[c].notna().any() else '' for c in df.columns]
})

column_summary

In [ ]:
categorical_like_cols = [c for c in df.columns if df[c].nunique(dropna=True) <= 25]
small_value_counts = {}
for col in categorical_like_cols:
    vc = df[col].fillna('MISSING').value_counts(dropna=False).head(25)
    small_value_counts[col] = {str(k): int(v) for k, v in vc.items()}

small_value_counts

In [ ]:
paper_summary = {
    'dataset_file': str(DATA_PATH),
    'total_rows': int(df.shape[0]),
    'total_columns': int(df.shape[1]),
    'column_names': df.columns.tolist(),
}

if 'crime_date' in df.columns:
    paper_summary['crime_date_min'] = str(df['crime_date'].min())
    paper_summary['crime_date_max'] = str(df['crime_date'].max())

for col in ['row_id', 'cmplnt_fr_tm_raw', 'cmplnt_fr_tm', 'occurred_dt', 'crime_time', 'latitude', 'longitude']:
    if col in df.columns:
        paper_summary[f'{col}_non_missing'] = int(df[col].notna().sum())

if 'crime_time_source' in df.columns:
    paper_summary['crime_time_source_counts'] = {
        str(k): int(v) for k, v in df['crime_time_source'].fillna('missing').value_counts(dropna=False).items()
    }

paper_summary

In [ ]:
overview_path = OUT_DIR / 'dataset_overview_v16c.csv'
columns_path = OUT_DIR / 'dataset_column_summary_v16c.csv'
json_path = OUT_DIR / 'dataset_methodology_summary_v16c.json'
value_counts_path = OUT_DIR / 'dataset_small_value_counts_v16c.json'

overview_df.to_csv(overview_path, index=False)
column_summary.to_csv(columns_path, index=False)
json_path.write_text(json.dumps(paper_summary, indent=2), encoding='utf-8')
value_counts_path.write_text(json.dumps(small_value_counts, indent=2), encoding='utf-8')

print('Saved overview to', overview_path)
print('Saved column summary to', columns_path)
print('Saved methodology JSON to', json_path)
print('Saved small value counts JSON to', value_counts_path)

In [ ]:
print('Methodology-ready quick summary')
print('- total rows:', df.shape[0])
print('- total columns:', df.shape[1])
if 'crime_date' in df.columns:
    print('- crime date range:', df['crime_date'].min(), 'to', df['crime_date'].max())
if 'row_id' in df.columns:
    print('- placeholder rows with row_id == 0:', int((df['row_id'] == 0).fillna(False).sum()))
if 'latitude' in df.columns and 'longitude' in df.columns:
    print('- non-missing latitude:', int(df['latitude'].notna().sum()))
    print('- non-missing longitude:', int(df['longitude'].notna().sum()))
if 'crime_time_source' in df.columns:
    print('- crime_time_source counts:')
    print(df['crime_time_source'].fillna('missing').value_counts(dropna=False))